# ABT 2 Creation: Main Churn Model `targetChurn`



In [0]:
%pip install feature_engine
%pip install xgboost lightgbm catboost
%pip install seaborn

In [0]:
# Tables
tb_fs = 'feature_store_train'
tb_ts = 'target_store_train'

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow

In [0]:
df_fs = spark.table(f'workspace.default.{tb_fs}').toPandas()
df_ts = spark.table(f'workspace.default.{tb_ts}').toPandas()

df = df_fs.merge(df_ts, on='customerID')
df = df.drop(columns=['CustomerFeedback_adj'])
df.head()


In [0]:
# df.describe().T
df.columns.tolist()

In [0]:
features = [
    # --- 1. Customer Demographics (Numeric Binary) ---
    'isMale',
    'SeniorCitizen',
    'isPartner',
    'HasDependents',
    'MonthlyIncome',

    # --- 2. Account & Tenure ---
    'nbrTenure',                  # Continuous tenure
    'isNewCustomer',              # High-risk early churn flag (tenure <= 6)
    'isPaperlessBilling',
    'isContractOneYear',          # Baseline = Month-to-month
    'isContractTwoYear',

    # --- 3. Payment Methods (One flag dropped as reference category) ---
    'isPaymentMethod_ElectronicCheck',  # Known highest-risk payment method
    'isPaymentMethod_MailedCheck',
    'isPaymentMethod_BankTransfer',
    'isPaymentMethod_CreditCard',

    # --- 4. Core Telecom Services ---
    'hasPhoneService',
    'hasMultipleLines',
    'hasInternetService',
    'isInternetService_DSL',
    'isInternetService_Fiber',          # Known high churn driver in EDA

    # --- 5. Add-on Internet Services ---
    'hasOnlineSecurity',
    'hasOnlineBackup',
    'hasDeviceProtection',
    'hasTechSupport',
    'hasStreamingTV',
    'hasStreamingMovies',
    'nbrInternetServicesAddOnQty',      # Total count of add-ons
    'hasSecurityBundle',                # High-loyalty flag

    # --- 6. High-Stakes Business Interactions (The Revenue Levers) ---
    'isM2M_dual',                       # The #1 revenue hemorrhage segment (47.3% churn)
    'isM2M_internetOnly',
    'isOneYear_dual',
    'isTwoYear_dual',

    # --- 7. Billing & Financial Ratios ---
    'MonthlyCharges',
    'TotalCharges',                     # Imputed by your champion Ridge model
    'TotalSpend',                       # Interaction (tenure * MonthlyCharges)
    'chargeRatio',                      # MonthlyCharges / MonthlyIncome (affordability stress)
    'numServices',                      # Service breadth
    'chargesPerService'                 # Value perception
]


In [0]:
from sklearn import model_selection

target = 'TargetChurn'

# drop_cols = ['customerID','tenure_calc', target]   # adjust to your actual non-feature columns
# features = [c for c in df.columns if c not in drop_cols]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f'Size Train dataset: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Target ratio: ({y_train.mean()*100:.2f}%)')

print(f'Size Test dataset: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Target ratio: ({y_test.mean()*100:.2f}%)')

In [0]:
from sklearn.pipeline import Pipeline
from feature_engine import imputation
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
import mlflow

# Impute TotalCharges using the MLflow Champion model
imputer_model = mlflow.sklearn.load_model("models:/workspace.default.total_charges_ridge_tuned@champion")

def impute_total_charges(df):
    X = df.copy()
    total = pd.to_numeric(X['TotalCharges'], errors='coerce')
    missing_mask = total.isna()
    if missing_mask.sum() > 0:
        features = ['isContractTwoYear', 'TotalSpend', 'isContractOneYear', 'nbrTenure', 'MonthlyCharges']
        total.loc[missing_mask] = imputer_model.predict(X.loc[missing_mask, features])
        
    X['TotalCharges'] = total
    return X

# Resolve tenure (Bruno formula: TotalCharges / MonthlyCharges)
def resolve_tenure(df):
    X = df.copy()
    # tenure_num = pd.to_numeric(X['tenure'], errors='coerce')
    tenure_num = pd.to_numeric(X['nbrTenue'], errors='coerce')
    total = pd.to_numeric(X['TotalCharges'], errors='coerce')
    monthly = pd.to_numeric(X['MonthlyCharges'], errors='coerce')
    
    # Calculate tenure if missing but TotalCharges exists
    mask_calc = tenure_num.isna() & total.notna()
    tenure_num.loc[mask_calc] = (total.loc[mask_calc] / monthly.loc[mask_calc]).round()

    return X



# Wrap functions as pipeline steps
impute_total_charges_step = FunctionTransformer(impute_total_charges)
impute_tenure_step = FunctionTransformer(resolve_tenure)


In [0]:
features_mean = ['nbrTenure']
feature_0 = ['isMale', 'isPartner', 'SeniorCitizen', 'hasDependents']

# Method Definition
impute_mean = imputation.MeanMedianImputer(imputation_method='mean', variables=features_mean)
impute_0 = imputation.CategoricalImputer(imputation_method='missing', variables=feature_0)

#



In [0]:
import mlflow
from sklearn import ensemble
from sklearn import linear_model
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn import pipeline
from sklearn import metrics

# Set the registry to Unity Catalog
# mlflow.set_experiment(experiment_id=1316272881415465)

with mlflow.start_run():

    mlflow.sklearn.autolog()
    
    clf = linear_model.LogisticRegression(random_state=42)
    # clf = ensemble.RandomForestClassifier(min_samples_leaf=30, n_estimators=500, random_state=42)
    # clf = ensemble.AdaBoostClassifier(n_estimators=1000, learning_rate=0.9, random_state=42)
    # clf = ensemble.GradientBoostingClassifier(n_estimators=1000, learning_rate=0.9, random_state=42)
    # clf = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, auto_class_weights='Balanced', random_state=42)
    # clf = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.9, random_state=42)
    # clf = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.9, random_state=42)

# # PIPELINE
model_pipeline = Pipeline(steps=[
    ('impute_total_charges', impute_total_charges_step),
    ('impute_tenure', impute_tenure_step),
    ('impute_mean', impute_mean),
    ('impute_0', impute_0),
    # ('classifier', LGBMClassifier(random_state=42, is_unbalance=True))
])

# 1. Grab all transformation steps (everything before 'classifier')
imputation_pipeline = model_pipeline
# 2. Transform test data or raw inference data
transformed_data = imputation_pipeline.transform(X_test[features])
# 3. Check for any missing values
if isinstance(transformed_data, np.ndarray):
    total_nulls = np.isnan(transformed_data).sum()
    print(f"Total NaNs remaining after pipeline: {total_nulls}")
    assert total_nulls == 0, f"FAILED: Found {total_nulls} NaNs in output!"
    print("SUCCESS: 0 NaNs found! The pipeline cleanly resolved all missing values.")
else:
    # If it returns a DataFrame
    null_counts = transformed_data.isna().sum()
    print("Missing values per column after pipeline:")
    print(null_counts[null_counts > 0])
    assert null_counts.sum() == 0, "FAILED: Some columns still have nulls!"
    print("SUCCESS: 0 NaNs found across all columns!")

In [0]:
import mlflow
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn import pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# REUSABLE CHURN MODEL FUNCTION
def train_and_log_churn(
    model_name,
    estimator,
    param_grid=None,          # None -> plain fit; dict -> GridSearchCV
    scaler='minmax',          # 'minmax', 'standard', or None
    cv=5,
    scoring='roc_auc',        # Optimizes for class ranking & separation
    X_train=None, y_train=None, X_test=None, y_test=None,
    features=None,
    max_tuning_runs=20,
    impute_0=None,
    impute_mean=None
):
    # MLflow autolog logs parameter tuning and child runs
    mlflow.sklearn.autolog(exclusive=False, max_tuning_runs=max_tuning_runs)

    with mlflow.start_run(run_name=model_name) as run:
        
        # 1. Log feature count
        n_features = len(features) if features is not None else X_train.shape[1]
        mlflow.log_param('n_features', n_features)

        # 2. Build Pipeline
        steps = [
            ('impute_total_charges', impute_total_charges_step),
            ('impute_tenure', impute_tenure_step),
            ('impute_mean', impute_mean),
            ('impute_0', impute_0),
        ]

        # Add optional Scaler
        if scaler == 'minmax':
            steps.append(('scaler', MinMaxScaler()))
        elif scaler == 'standard':
            steps.append(('scaler', StandardScaler()))

        # Add Classifier
        steps.append(('classifier', estimator))
        model_pipeline = pipeline.Pipeline(steps=steps)

        # 3. Fit: Plain Fit OR GridSearchCV
        if param_grid is not None:
            # Prefix parameters with 'classifier__' so GridSearchCV tunes the classifier step inside the pipeline
            prefixed_params = {
                f"classifier__{k}": v for k, v in param_grid.items()
            }
            
            strat_cv = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
            grid = GridSearchCV(
                estimator=model_pipeline,
                param_grid=prefixed_params,
                cv=strat_cv,
                scoring=scoring,
                n_jobs=-1,
            )
            grid.fit(X_train[features], y_train)
            best_pipeline = grid.best_estimator_
            
            mlflow.log_params(grid.best_params_)
            mlflow.log_metric('best_cv_score', grid.best_score_)
            print(f"[{model_name}] Best Params: {grid.best_params_}")
            print(f"[{model_name}] Best CV {scoring}: {grid.best_score_:.4f}")
        else:
            model_pipeline.fit(X_train[features], y_train)
            best_pipeline = model_pipeline

        # 4. Predict Probabilities & Compute Business Cutoff
        y_train_prob = best_pipeline.predict_proba(X_train[features])[:, 1]
        y_test_prob = best_pipeline.predict_proba(X_test[features])[:, 1]
        
        # Base rate cutoff for retention decisions
        base_rate_cutoff = float(y_train.mean())
        mlflow.log_param("base_rate_cutoff", round(base_rate_cutoff, 4))

        y_train_pred = (y_train_prob >= 0.50).astype(int)
        y_test_pred = (y_test_prob >= 0.50).astype(int)
        y_test_pred_cutoff = (y_test_prob >= base_rate_cutoff).astype(int)

        # 5. Metrics Calculation
        auc_train = metrics.roc_auc_score(y_train, y_train_prob)
        auc_test = metrics.roc_auc_score(y_test, y_test_prob)
        acc_train = metrics.accuracy_score(y_train, y_train_pred)
        acc_test = metrics.accuracy_score(y_test, y_test_pred)
        rec_test_0_5 = metrics.recall_score(y_test, y_test_pred)
        rec_test_cutoff = metrics.recall_score(y_test, y_test_pred_cutoff)
        f1_test_cutoff = metrics.f1_score(y_test, y_test_pred_cutoff)
        lift_auc_test = (auc_test / 0.50 - 1) * 100

        print(f"=== {model_name} Results ===")
        print(f"Train AUC: {auc_train:.4f} | Test AUC: {auc_test:.4f}")
        print(f"Test Recall (0.50): {rec_test_0_5:.4f} | Test Recall (Corte {base_rate_cutoff:.2f}): {rec_test_cutoff:.4f}")
        print(f"Lift sobre Aleatório: +{lift_auc_test:.2f}%\n")

        # 6. Log Metrics
        mlflow.log_metrics({
            'auc_train': auc_train,
            'auc_test': auc_test,
            'acc_train': acc_train,
            'acc_test': acc_test,
            'recall_test_0_5': rec_test_0_5,
            'recall_test_cutoff': rec_test_cutoff,
            'f1_test_cutoff': f1_test_cutoff,
            'lift_auc_test_pct': lift_auc_test
        })

        # 7. Register Model in Unity Catalog / Model Registry
        mlflow.sklearn.log_model(
            sk_model=best_pipeline,
            artifact_path="churn_model",
            registered_model_name=f"workspace.default.{model_name}",
            input_example=X_train[features].iloc[[0]]
        )

        print(f"Model {model_name} registered successfully!")
        return best_pipeline

# Models

In [0]:
import mlflow
from sklearn import ensemble, linear_model
import lightgbm as lgb
import xgboost as xgb

# Set Experiment
mlflow.set_experiment(experiment_id=1203535636207065)

In [0]:
# Baseline: Logistic Regression (with Tuning)
train_and_log_churn(
    model_name="churn_logistic_regression",
    estimator=linear_model.LogisticRegression(max_iter=1000, random_state=42),
    param_grid={'C': [0.01, 0.1, 1.0, 10.0]},
    scaler='minmax',
    cv=5,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    features=features, impute_0=impute_0, impute_mean=impute_mean
)

In [0]:
# Random Forest (with Tuning)
train_and_log_churn(
    model_name="churn_random_forest",
    estimator=ensemble.RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid={
        'n_estimators': [200, 500],
        'min_samples_leaf': [20, 30],
        'max_depth': [8, 12]
    },
    cv=5,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    features=features, impute_0=impute_0, impute_mean=impute_mean
)

In [0]:
# LightGBM (Champion Candidate with Tuning)
train_and_log_churn(
    model_name="churn_lightgbm_tuned",
    estimator=lgb.LGBMClassifier(random_state=42, is_unbalance=True, n_jobs=-1),
    param_grid={
        'n_estimators': [100, 300],
        'learning_rate': [0.03, 0.08],
        'num_leaves': [15, 31]
    },
    cv=5,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    features=features, impute_0=impute_0, impute_mean=impute_mean
)